# TRPG Direct Play

这个 notebook 用来直接体验游戏流程。

- 不展示调试中间态和上下文构建细节
- 每回合直接展示 `Dicer / NPC Manager / Narrator / Director` 的输出
- 使用 `run_turn(background_director=False)`，这样你每回合都能看到本轮生成的 Director 输出
- 需要在循环中直接输入玩家发言


In [ ]:
import sys
import json
import importlib
from datetime import datetime
from pathlib import Path
from typing import Any

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if not (candidate / 'Code').exists():
            continue
        if (candidate / 'Story').exists() or (candidate / 'story').exists() or (candidate / 'Prompt').exists():
            return candidate
    raise RuntimeError(f'Unable to locate repo root from: {start}')


ROOT = find_repo_root(Path.cwd().resolve())
CODE_DIR = (ROOT / 'Code').resolve()
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

import trpg_runtime.models
import trpg_runtime.prompt_loader
import trpg_runtime.state_store
import trpg_runtime.engine
import trpg_runtime

importlib.reload(trpg_runtime.models)
importlib.reload(trpg_runtime.prompt_loader)
importlib.reload(trpg_runtime.state_store)
importlib.reload(trpg_runtime.engine)
importlib.reload(trpg_runtime)

from trpg_runtime import MinimalTRPGEngine, PromptRepository, RuntimeLogEvent, RuntimeOptions


def to_plain_data(value: Any) -> Any:
    if hasattr(value, 'model_dump'):
        return value.model_dump(mode='python')
    return value


def print_heading(title: str) -> None:
    print(f"\n{'=' * 18} {title} {'=' * 18}")


def print_json_block(title: str, value: Any) -> None:
    print_heading(title)
    print(json.dumps(to_plain_data(value), ensure_ascii=False, indent=2))


def print_text_block(title: str, text: str) -> None:
    print_heading(title)
    print(text)


def print_progress(message: str) -> None:
    print(f'[LOG] {message}')


def print_log_event(event: RuntimeLogEvent) -> None:
    print(f'[LOG][{event.phase}/{event.stage}] {event.message}')


def print_turn_outputs(turn_index: int, result) -> None:
    print_heading(f'第 {turn_index} 回合结果总览')
    print(f"玩家输入: {result.action.raw_text}")
    print(f"本轮 Narrator 使用的 Director tone: {result.director_state_used.current_tone}")
    print_json_block('Dicer Output', result.dicer_result)
    print_json_block('NPC Manager Output', result.npc_result)
    print_text_block('Narrator Output', result.narration)
    print_json_block('Director State Used This Turn', result.director_state_used)
    print_json_block('Director Output For Next Turn', result.next_director_result)


def save_agent_transcript(root: Path, engine, rule_code: str, story_code: str, turn_logs: list[dict[str, object]]) -> Path:
    output_dir = root / 'test' / 'play_logs'
    output_dir.mkdir(parents=True, exist_ok=True)
    session_id = getattr(engine.state.meta, 'session_id', 'session')
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    output_path = output_dir / f'{rule_code}_{story_code}_{session_id}_{timestamp}.txt'

    lines = [
        f'Rule: {rule_code}',
        f'Story: {story_code}',
        f'Session: {session_id}',
        f'Turn Count: {len(turn_logs)}',
        '',
    ]

    for record in turn_logs:
        result = record['turn_result']
        lines.extend([
            '=' * 28,
            f"Turn {record['turn_index']}",
            '=' * 28,
            f"Player Input: {record['player_text']}",
            '',
            '[Dicer Output]',
            json.dumps(to_plain_data(result.dicer_result), ensure_ascii=False, indent=2),
            '',
            '[NPC Manager Output]',
            json.dumps(to_plain_data(result.npc_result), ensure_ascii=False, indent=2),
            '',
            '[Narrator Output]',
            result.narration,
            '',
            '[Director State Used This Turn]',
            json.dumps(to_plain_data(result.director_state_used), ensure_ascii=False, indent=2),
            '',
            '[Director Output For Next Turn]',
            json.dumps(to_plain_data(result.next_director_result), ensure_ascii=False, indent=2) if result.next_director_result is not None else 'None',
            '',
        ])

    output_path.write_text('\n'.join(lines), encoding='utf-8')
    return output_path


print('运行环境已加载。')


## 默认配置

如果你想换规则或剧本，直接改下面这一格后重新运行。

In [ ]:
RULE_CODE = 'DET'
STORY_CODE = 'THE_FIRSTMURDER'
PLAYER_NAME = '调查员'
MAX_TURNS = 200

OPTIONS = RuntimeOptions(
    full_rule_text_for_agents=True,
    full_story_text_for_agents=True,
    dicer_dialogue_window=2,
    npc_dialogue_window=3,
    narrator_dialogue_window=5,
    director_dialogue_window=5,
    max_dialogue_window=5,
)

print({
    'RULE_CODE': RULE_CODE,
    'STORY_CODE': STORY_CODE,
    'PLAYER_NAME': PLAYER_NAME,
    'MAX_TURNS': MAX_TURNS,
})


## 创建引擎并生成开场

这一格会直接走完整初始化流程：读取 state 文件，缺失时自动解析并落盘，然后生成 beginning/opening。

In [ ]:
try:
    engine.shutdown(wait=False)
except Exception:
    pass

repo = PromptRepository()
engine = MinimalTRPGEngine.from_prompt_files(
    rule_code=RULE_CODE,
    story_code=STORY_CODE,
    player_name=PLAYER_NAME,
    prompt_repository=repo,
    options=OPTIONS,
)

opening = engine.generate_opening(event_callback=print_log_event)

print(f'已载入规则: {RULE_CODE}')
print(f'已载入剧本: {STORY_CODE}')
print_text_block('Opening', opening)
print('\n可用命令: /state, /director, /quit')


## 直接开玩

这一格会进入真实回合循环。

- 直接输入你要说的话
- `/state` 查看当前完整 state
- `/director` 查看当前已提交 DirectorState
- `/quit` 结束本轮体验，并自动导出本次游玩的 agent 输出


In [ ]:
turn_index = 1
turn_logs = []
transcript_path = None

while turn_index <= MAX_TURNS:
    player_text = input(f'[{turn_index}/{MAX_TURNS}] 你要说什么？ ').strip()
    if not player_text:
        print('请输入内容。')
        continue

    if player_text == '/quit':
        if turn_logs:
            transcript_path = save_agent_transcript(ROOT, engine, RULE_CODE, STORY_CODE, turn_logs)
            print(f'游戏循环已结束，记录已保存到: {transcript_path}')
        else:
            print('游戏循环已结束，本次没有可保存的回合记录。')
        break

    if player_text == '/state':
        print_json_block('Current GameState', engine.state)
        continue

    if player_text == '/director':
        print_json_block('Current DirectorState', engine.state.director)
        continue

    print_progress(f'Starting turn {turn_index}...')
    result = engine.run_turn(
        player_text,
        background_director=False,
        event_callback=print_log_event,
    )
    turn_logs.append({
        'turn_index': turn_index,
        'player_text': player_text,
        'turn_result': result,
    })
    print_turn_outputs(turn_index, result)
    print_progress(f'Turn {turn_index} finished.')
    turn_index += 1

if turn_index > MAX_TURNS:
    if turn_logs:
        transcript_path = save_agent_transcript(ROOT, engine, RULE_CODE, STORY_CODE, turn_logs)
        print(f'已达到最大回合数，记录已保存到: {transcript_path}')
    else:
        print('已达到最大回合数。')
